# <font color="#418FDE" size="6.5" uppercase>**Logistic Regression**</font>

>Last update: 20260710.
    
By the end of this Lecture, you will be able to:
- Explain how logistic regression converts engineering features into fault or pass-fail probabilities. 
- Implement sigmoid prediction and gradient-based training for logistic regression from scratch. 
- Evaluate classification thresholds using confusion matrix, precision, recall, and F1 score. 


## **1. Probability Outputs**

### **1.1. Sigmoid Probability Mapping**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Mechanical Engineers/Module_05/Lecture_A/image_01_01.jpg?v=1783666003" width="250">



>* Engineering measurements become one evidence score
>* Sigmoid converts scores into fault probabilities

>* Positive scores mean higher predicted risk
>* Sensor data becomes understandable fault probability

>* Most responsive for uncertain borderline cases
>* Extreme probabilities change more gradually



In [ ]:
#@title Python Code - Sigmoid Probability Mapping

# Logistic regression maps evidence into probabilities.
# Sigmoid curves support engineering risk decisions.
# This example uses small synthetic measurements.

# Import numerical and plotting libraries.
import numpy as np
import matplotlib.pyplot as plt

# Define the sigmoid probability mapping.
def sigmoid(score):
    clipped_score = np.clip(score, -40, 40)
    return 1 / (1 + np.exp(-clipped_score))

# Create engineering evidence scores.
scores = np.linspace(-8, 8, 161)
probabilities = sigmoid(scores)

# Validate matching vector shapes.
assert scores.shape == probabilities.shape
assert probabilities.size == 161

# Define three inspection cases.
case_names = ["safe", "borderline", "faulty"]
case_scores = np.array([-4.0, 0.3, 3.2])
case_probs = sigmoid(case_scores)

# Print compact probability interpretations.
print("Sigmoid maps any score into 0 to 1.")
print(f"Safe case probability: {case_probs[0]:.3f}")
print(f"Borderline probability: {case_probs[1]:.3f}")
print(f"Faulty case probability: {case_probs[2]:.3f}")

# Plot the sigmoid probability curve.
plt.figure(figsize=(7, 4))
plt.plot(scores, probabilities, linewidth=2)

# Add the engineering cases.
plt.scatter(case_scores, case_probs, s=70)
for name, x_value, y_value in zip(case_names, case_scores, case_probs):
    plt.text(x_value + 0.15, y_value, name)

# Add decision reference lines.
plt.axhline(0.5, linestyle="--", color="gray")
plt.axvline(0.0, linestyle="--", color="gray")

# Label the probability mapping plot.
plt.title("Sigmoid Probability Mapping")
plt.xlabel("Combined engineering evidence score")
plt.ylabel("Predicted fault probability")

# Display the single teaching plot.
plt.ylim(-0.05, 1.05)
plt.grid(True, alpha=0.3)
plt.show()



### **1.2. Sigmoid Probability Mapping**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Mechanical Engineers/Module_05/Lecture_A/image_01_02.jpg?v=1783666013" width="250">



>* Engineering measurements become fault likelihood estimates
>* Sigmoid converts scores into bounded probabilities

>* Extreme scores give near-zero or near-one probabilities
>* Middle probabilities flag uncertain borderline cases

>* Probabilities guide risk-based engineering decisions
>* Sigmoid outputs support actions beyond labels



In [ ]:
#@title Python Code - Sigmoid Probability Mapping

# Logistic regression maps scores into probabilities.
# Sigmoid outputs support engineering risk decisions.
# This example uses simple weld measurements.

# Import numerical and plotting libraries.
import numpy as np
import matplotlib.pyplot as plt

# Define a stable sigmoid function.
def sigmoid(score_values):
    clipped_values = np.clip(score_values, -50, 50)
    probability_values = 1 / (1 + np.exp(-clipped_values))
    return probability_values

# Create example engineering feature measurements.
weld_temperature_c = np.array([610, 640, 670, 700, 730])
cooling_time_s = np.array([42, 38, 34, 30, 26])
ultrasonic_signal = np.array([0.10, 0.25, 0.45, 0.70, 0.90])

# Check that feature arrays align correctly.
feature_lengths = [len(weld_temperature_c), len(cooling_time_s), len(ultrasonic_signal)]
assert len(set(feature_lengths)) == 1, "Feature lengths must match."

# Standardize features for comparable score contributions.
temp_scaled = (weld_temperature_c - 670) / 40
cool_scaled = (34 - cooling_time_s) / 8
signal_scaled = (ultrasonic_signal - 0.45) / 0.25

# Combine features into one logistic score.
score_values = -0.20 + 0.90 * temp_scaled + 0.70 * cool_scaled + 1.10 * signal_scaled
probability_values = sigmoid(score_values)
class_labels = probability_values >= 0.50

# Print compact probability interpretation results.
print("Sigmoid maps any score into a 0 to 1 probability.")
print("Case | score | defect probability | predicted label")
for index_value in range(len(score_values)):
    label_text = "defect" if class_labels[index_value] else "pass"
    print(f"{index_value + 1:>4} | {score_values[index_value]:>5.2f} | {probability_values[index_value]:>18.2f} | {label_text}")

# Plot the sigmoid curve and example weld scores.
curve_scores = np.linspace(-6, 6, 200)
curve_probabilities = sigmoid(curve_scores)
plt.figure(figsize=(7, 4))
plt.plot(curve_scores, curve_probabilities, label="sigmoid mapping")
plt.scatter(score_values, probability_values, color="red", label="weld cases")
plt.axhline(0.50, color="gray", linestyle="--", label="0.50 threshold")
plt.xlabel("combined feature score")
plt.ylabel("predicted defect probability")
plt.title("Logistic Regression Sigmoid Probability Mapping")
plt.legend()
plt.grid(True, alpha=0.30)
plt.show()



### **1.3. Decision Boundary**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Mechanical Engineers/Module_05/Lecture_A/image_01_03.jpg?v=1783666005" width="250">



>* Boundary separates predicted classes by probability threshold
>* Engineering measurements determine fault or normal side

>* Feature representation shapes the boundary
>* Probabilities change smoothly near ambiguous cases

>* Thresholds reflect error costs and goals
>* Boundaries turn probabilities into practical decisions



In [ ]:
#@title Python Code - Decision Boundary

# Logistic regression creates smooth probability boundaries.
# This example uses pump inspection measurements.
# Threshold choices move the decision boundary.

# Import numerical and plotting tools.
import numpy as np
import matplotlib.pyplot as plt

# Create a small deterministic feature grid.
vibration = np.linspace(0.2, 2.8, 80)
temperature = np.linspace(45.0, 105.0, 80)
V, T = np.meshgrid(vibration, temperature)

# Define simple logistic regression coefficients.
bias = -8.0
weight_vibration = 2.4
weight_temperature = 0.075

# Convert engineering features into linear scores.
scores = bias + weight_vibration * V + weight_temperature * T
prob_fault = 1.0 / (1.0 + np.exp(-scores))

# Validate the probability grid shape.
assert prob_fault.shape == V.shape
assert np.all((prob_fault >= 0.0) & (prob_fault <= 1.0))

# Compare two practical probability thresholds.
threshold_standard = 0.50
threshold_cautious = 0.30

# Compute boundary temperatures for each threshold.
logit_standard = np.log(threshold_standard / (1.0 - threshold_standard))
logit_cautious = np.log(threshold_cautious / (1.0 - threshold_cautious))

# Rearrange the score equation into boundary lines.
boundary_standard = (logit_standard - bias - weight_vibration * vibration) / weight_temperature
boundary_cautious = (logit_cautious - bias - weight_vibration * vibration) / weight_temperature

# Print a compact interpretation for learners.
print("Model output: probability of pump fault.")
print("Boundary means probability equals chosen threshold.")
print("Lower threshold flags more borderline pumps.")
print(f"At vibration 1.5 in/s, standard boundary is {np.interp(1.5, vibration, boundary_standard):.1f} C.")
print(f"At vibration 1.5 in/s, cautious boundary is {np.interp(1.5, vibration, boundary_cautious):.1f} C.")

# Plot probabilities and both decision boundaries.
plt.figure(figsize=(7, 5))
contour = plt.contourf(V, T, prob_fault, levels=20, cmap="RdYlBu_r")
plt.colorbar(contour, label="Predicted fault probability")

# Add threshold lines to show operating choices.
plt.plot(vibration, boundary_standard, color="black", linewidth=2, label="Threshold 0.50")
plt.plot(vibration, boundary_cautious, color="purple", linewidth=2, linestyle="--", label="Threshold 0.30")
plt.xlabel("Vibration level, inches per second")

# Finish the plot with engineering context.
plt.ylabel("Operating temperature, degrees Celsius")
plt.title("Logistic Regression Decision Boundary")
plt.ylim(45, 105)
plt.legend()
plt.show()



## **2. Manual Logistic Training**

### **2.1. Sigmoid Prediction**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Mechanical Engineers/Module_05/Lecture_A/image_02_01.jpg?v=1783665994" width="250">



>* Linear feature scores become sigmoid probabilities
>* Near zero means normal, near one fault

>* Sensitive near uncertain decision boundaries
>* Stable extremes show stronger model confidence

>* Forward pass converts features into probabilities
>* Training adjusts weights using prediction errors



In [ ]:
#@title Python Code - Sigmoid Prediction

# Demonstrate sigmoid prediction for engineering classification.
# Use tiny sensor data and manual weights.
# Convert raw scores into fault probabilities.

import numpy as np
import matplotlib.pyplot as plt

# Store vibration and temperature features.
features = np.array([
    [0.10, 0.20],
    [0.40, 0.50],
    [0.80, 0.70],
    [1.20, 1.00]
])

# Validate the feature matrix shape.
if features.shape != (4, 2):
    raise ValueError("Expected four rows and two features")

# Define manual logistic regression parameters.
weights = np.array([3.0, 2.0])
bias = -3.0

# Define a numerically stable sigmoid function.
def sigmoid(raw_scores):
    clipped_scores = np.clip(raw_scores, -50, 50)
    return 1.0 / (1.0 + np.exp(-clipped_scores))

# Compute raw linear scores first.
raw_scores = features @ weights + bias
probabilities = sigmoid(raw_scores)

# Convert probabilities using a threshold.
threshold = 0.50
predicted_faults = probabilities >= threshold

# Print compact teaching results.
print("NumPy version:", np.__version__)
print("Rows represent increasing vibration and temperature.")
print("Raw scores:", np.round(raw_scores, 2).tolist())
print("Fault probabilities:", np.round(probabilities, 3).tolist())
print("Predicted faults:", predicted_faults.astype(int).tolist())

# Plot sigmoid curve and example scores.
x_values = np.linspace(-6, 6, 200)
y_values = sigmoid(x_values)

# Create one clear sigmoid visualization.
plt.figure(figsize=(7, 4))
plt.plot(x_values, y_values, label="sigmoid(score)")
plt.scatter(raw_scores, probabilities, color="red", label="machines")
plt.axhline(threshold, color="gray", linestyle="--", label="threshold")
plt.xlabel("Raw linear score")
plt.ylabel("Predicted fault probability")
plt.title("Sigmoid Prediction From Manual Logistic Scores")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()



### **2.2. Gradient Descent Training**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Mechanical Engineers/Module_05/Lecture_A/image_02_02.jpg?v=1783666001" width="250">



>* Learns weights from labeled examples
>* Moves parameters to reduce prediction error

>* Loss highlights confident wrong predictions
>* Weights adjust based on feature errors

>* Repeated updates gradually improve class separation
>* Tune learning rate and monitor validation performance



In [ ]:
#@title Python Code - Gradient Descent Training

# Manual logistic training uses gradient descent.
# Tiny engineering data keeps learning visible.
# We train weights without ML libraries.

import numpy as np
import matplotlib.pyplot as plt

# Set deterministic numeric display and randomness.
np.random.seed(7)
np.set_printoptions(precision=3, suppress=True)

# Create two engineering features for parts.
vibration = np.array([0.8, 1.0, 1.2, 1.5, 2.0, 2.4, 2.8, 3.2])
temperature = np.array([45, 48, 50, 55, 62, 68, 72, 80])
failed = np.array([0, 0, 0, 0, 1, 1, 1, 1])

# Stack features into one design matrix.
X_raw = np.column_stack((vibration, temperature))
y = failed.reshape(-1, 1)

# Validate the tiny training table shape.
assert X_raw.shape == (8, 2)
assert y.shape == (8, 1)

# Standardize features for stable gradient descent.
means = X_raw.mean(axis=0, keepdims=True)
scales = X_raw.std(axis=0, keepdims=True)
X = (X_raw - means) / scales

# Add a bias column for intercept learning.
ones = np.ones((X.shape[0], 1))
Xb = np.column_stack((ones, X))

# Define sigmoid probability conversion.
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

# Initialize weights and training settings.
weights = np.zeros((Xb.shape[1], 1))
learning_rate = 0.4
steps = 120
loss_history = []

# Train logistic regression using batch gradients.
for step in range(steps):
    probabilities = sigmoid(Xb @ weights)
    errors = probabilities - y
    gradient = (Xb.T @ errors) / len(y)

    weights = weights - learning_rate * gradient
    clipped = np.clip(probabilities, 1e-9, 1 - 1e-9)
    loss = -np.mean(y * np.log(clipped) + (1 - y) * np.log(1 - clipped))
    loss_history.append(loss)

# Convert final probabilities into pass fail predictions.
final_probabilities = sigmoid(Xb @ weights)
predicted_failed = (final_probabilities >= 0.5).astype(int)
accuracy = np.mean(predicted_failed == y)

# Print a compact training summary.
print(f"NumPy version: {np.__version__}")
print(f"Initial loss: {loss_history[0]:.3f}")
print(f"Final loss: {loss_history[-1]:.3f}")
print(f"Learned weights: {weights.ravel()}")
print(f"Training accuracy: {accuracy:.2f}")

# Plot loss decrease during gradient descent.
plt.figure(figsize=(6, 4))
plt.plot(loss_history, color="navy", linewidth=2)
plt.xlabel("Gradient descent step")
plt.ylabel("Logistic loss")
plt.title("Manual Logistic Regression Training")
plt.grid(True, alpha=0.3)
plt.show()



### **2.3. Gradient Descent Updates**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Mechanical Engineers/Module_05/Lecture_A/image_02_03.jpg?v=1783665996" width="250">



>* Prediction errors guide weight and intercept changes
>* Many small updates gradually improve classification

>* Gradient sets update direction
>* Learning rate balances speed and stability

>* Updates shrink as predictions improve
>* Monitor loss to guide training adjustments



In [ ]:
#@title Python Code - Gradient Descent Updates

# Manual logistic training uses small engineering data.
# Gradient descent updates weights using prediction errors.
# This example stays fully built from scratch.

import numpy as np
import matplotlib.pyplot as plt

# Set deterministic behavior for reproducible teaching results.
np.random.seed(7)

# Create tiny sensor features for inspected machines.
temperature_c = np.array([62, 65, 67, 70, 73, 76, 80, 84], dtype=float)
vibration_mm = np.array([1.1, 1.3, 1.5, 1.8, 2.2, 2.6, 3.1, 3.7], dtype=float)

# Labels mark whether each machine failed inspection.
y = np.array([0, 0, 0, 0, 1, 1, 1, 1], dtype=float)

# Stack features into a two column design matrix.
X_raw = np.column_stack((temperature_c, vibration_mm))
assert X_raw.shape == (8, 2)

# Standardize features so updates have comparable scales.
feature_mean = X_raw.mean(axis=0)
feature_std = X_raw.std(axis=0)
X = (X_raw - feature_mean) / feature_std

# Define sigmoid for converting scores into probabilities.
def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

# Initialize logistic regression parameters from scratch.
weights = np.zeros(X.shape[1])
bias = 0.0
learning_rate = 0.35

# Store loss values for a compact training curve.
loss_history = []

# Run a short gradient descent training loop.
for step in range(80):
    scores = X @ weights + bias
    probabilities = sigmoid(scores)
    errors = probabilities - y

    grad_w = (X.T @ errors) / len(y)
    grad_b = errors.mean()
    weights = weights - learning_rate * grad_w
    bias = bias - learning_rate * grad_b

    clipped = np.clip(probabilities, 1e-9, 1 - 1e-9)
    loss = -np.mean(y * np.log(clipped) + (1 - y) * np.log(1 - clipped))
    loss_history.append(loss)

# Convert final probabilities into pass fail predictions.
final_probabilities = sigmoid(X @ weights + bias)
predictions = (final_probabilities >= 0.5).astype(int)
accuracy = (predictions == y).mean()

# Print only compact teaching results.
print(f"NumPy version: {np.__version__}")
print(f"Initial loss: {loss_history[0]:.3f}")
print(f"Final loss: {loss_history[-1]:.3f}")
print(f"Learned weights: {weights.round(3)}")
print(f"Learned bias: {bias:.3f}")
print(f"Training accuracy: {accuracy:.2f}")

# Plot loss to show updates improving the model.
plt.figure(figsize=(6, 4))
plt.plot(loss_history, color="navy", linewidth=2)
plt.xlabel("Gradient descent step")
plt.ylabel("Logistic loss")
plt.title("Manual Logistic Regression Training")
plt.grid(True, alpha=0.3)
plt.show()



## **3. Fault Classification Metrics**

### **3.1. Confusion Matrix**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Mechanical Engineers/Module_05/Lecture_A/image_03_01.jpg?v=1783665985" width="250">



>* Compares thresholded predictions with true labels
>* Reveals errors hidden by accuracy alone

>* Four outcomes show prediction consequences
>* Costs depend on the application

>* Threshold changes shift fault detection trade-offs
>* Confusion matrices guide application-specific decisions



In [ ]:
#@title Python Code - Confusion Matrix

# Confusion matrices summarize classification decision outcomes.
# Thresholds convert probabilities into predicted labels.
# Engineering faults often require careful threshold choices.

import numpy as np
import matplotlib.pyplot as plt

# Store true fault labels for inspected components.
y_true = np.array([0, 0, 1, 1, 0, 1, 0, 1])

# Store logistic regression fault probabilities.
y_prob = np.array([0.10, 0.35, 0.40, 0.82, 0.55, 0.70, 0.20, 0.48])

# Choose two thresholds for comparison.
thresholds = [0.50, 0.30]

# Define a small confusion matrix helper.
def confusion_counts(y_true, y_prob, threshold):
    y_pred = (y_prob >= threshold).astype(int)
    tp = int(np.sum((y_true == 1) & (y_pred == 1)))
    tn = int(np.sum((y_true == 0) & (y_pred == 0)))

    fp = int(np.sum((y_true == 0) & (y_pred == 1)))
    fn = int(np.sum((y_true == 1) & (y_pred == 0)))
    return tp, tn, fp, fn

# Print compact results for each threshold.
for threshold in thresholds:
    tp, tn, fp, fn = confusion_counts(y_true, y_prob, threshold)
    print(f"Threshold {threshold:.2f}: TP={tp}, TN={tn}, FP={fp}, FN={fn}")

# Build matrix for the first threshold.
tp, tn, fp, fn = confusion_counts(y_true, y_prob, thresholds[0])
matrix = np.array([[tn, fp], [fn, tp]])

# Plot one confusion matrix heatmap.
fig, ax = plt.subplots(figsize=(4, 3))
image = ax.imshow(matrix, cmap="Blues")

# Label rows and columns clearly.
ax.set_xticks([0, 1], labels=["Pred pass", "Pred fault"])
ax.set_yticks([0, 1], labels=["True pass", "True fault"])

# Add counts inside each cell.
for row in range(2):
    for col in range(2):
        ax.text(col, row, matrix[row, col], ha="center", va="center")

# Add a concise engineering title.
ax.set_title("Confusion Matrix at Threshold 0.50")
fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)

# Display the single teaching plot.
plt.tight_layout()
plt.show()



### **3.2. Precision**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Mechanical Engineers/Module_05/Lecture_A/image_03_02.jpg?v=1783665992" width="250">



>* Measures correctness of predicted faults
>* Useful when false alarms are costly

>* Threshold choice changes precision and false alarms
>* Higher thresholds trade missed faults for confidence

>* Judge precision within the application context
>* Compare with recall, F score, confusion matrix



In [ ]:
#@title Python Code - Precision

# Precision measures trustworthy fault alarms.
# Thresholds change positive prediction counts.
# Engineers compare alerts with reality.

import numpy as np

# Store predicted fault probabilities.
probabilities = np.array([0.92, 0.81, 0.74, 0.63, 0.55, 0.48, 0.39, 0.31])

# Store actual fault labels.
actual_faults = np.array([1, 1, 0, 1, 0, 0, 1, 0])

# Validate matching vector lengths.
assert probabilities.shape == actual_faults.shape

# Define thresholds for comparison.
thresholds = [0.50, 0.70]

# Print a compact heading.
print("Precision compares correct fault alerts with all fault alerts.")

# Evaluate each classification threshold.
for threshold in thresholds:
    predicted_faults = (probabilities >= threshold).astype(int)
    true_positive = int(np.sum((predicted_faults == 1) & (actual_faults == 1)))
    false_positive = int(np.sum((predicted_faults == 1) & (actual_faults == 0)))

    # Compute precision safely.
    predicted_positive = true_positive + false_positive
    precision = true_positive / predicted_positive if predicted_positive else 0.0
    print(f"Threshold {threshold:.2f}: TP={true_positive}, FP={false_positive}, precision={precision:.2f}")

# Explain the engineering interpretation.
print("Higher precision means fewer unnecessary maintenance alarms.")



### **3.3. F Score**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Mechanical Engineers/Module_05/Lecture_A/image_03_03.jpg?v=1783665987" width="250">



>* Combines precision and recall into one score
>* Balances false alarms against missed faults

>* F1 balances precision and recall equally
>* Threshold choice affects rare fault detection

>* Use F score for important positive cases
>* Choose thresholds based on real-world costs



In [ ]:
#@title Python Code - F Score

# F1 score balances alarms and missed faults.
# Thresholds change precision, recall, and F1.
# This example uses pump inspection probabilities.

import numpy as np
import matplotlib.pyplot as plt

# Store true pump fault labels.
y_true = np.array([0, 0, 1, 0, 1, 1, 0, 1])

# Store logistic regression fault probabilities.
y_prob = np.array([0.10, 0.35, 0.42, 0.55, 0.62, 0.70, 0.80, 0.90])

# Confirm matching vector lengths.
assert y_true.shape == y_prob.shape

# Define candidate alarm thresholds.
thresholds = np.array([0.30, 0.50, 0.70])

# Prepare metric storage lists.
precisions, recalls, f1_scores = [], [], []

# Evaluate each threshold manually.
for threshold in thresholds:
    y_pred = (y_prob >= threshold).astype(int)
    tp = int(np.sum((y_true == 1) & (y_pred == 1)))
    fp = int(np.sum((y_true == 0) & (y_pred == 1)))

    fn = int(np.sum((y_true == 1) & (y_pred == 0)))
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0

    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    precisions.append(precision)
    recalls.append(recall)
    f1_scores.append(f1)

# Print compact threshold comparison.
print("Threshold | Precision | Recall | F1 score")

# Show one concise row per threshold.
for threshold, precision, recall, f1 in zip(thresholds, precisions, recalls, f1_scores):
    print(f"{threshold:8.2f} | {precision:9.2f} | {recall:6.2f} | {f1:8.2f}")

# Identify the best F1 threshold.
best_index = int(np.argmax(f1_scores))

# Summarize the practical decision.
print(f"Best F1 threshold: {thresholds[best_index]:.2f}")

# Plot F1 score versus threshold.
plt.figure(figsize=(6, 4))
plt.plot(thresholds, f1_scores, marker="o", label="F1 score")
plt.plot(thresholds, precisions, marker="s", label="Precision")
plt.plot(thresholds, recalls, marker="^", label="Recall")

# Label the engineering tradeoff plot.
plt.xlabel("Fault alarm threshold")
plt.ylabel("Metric value")
plt.title("Pump Fault Classification Metrics")
plt.ylim(0, 1.05)

# Add grid and legend.
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()



# <font color="#418FDE" size="6.5" uppercase>**Logistic Regression**</font>


In this lecture, you learned to:
- Explain how logistic regression converts engineering features into fault or pass-fail probabilities. 
- Implement sigmoid prediction and gradient-based training for logistic regression from scratch. 
- Evaluate classification thresholds using confusion matrix, precision, recall, and F1 score. 

In the next Lecture (Lecture B), we will go over 'Nearest Neighbors'